# ACE 1O86 redocking validation

1O86 is human testis ACE co-crystallized with lisinopril (`LPR`). The active-site Zn²⁺ is retained in the receptor; the crystallographic LPR centroid defines the docking box. Run this before candidate docking.

**Setup notes (verified against Meeko 0.8 / Vina 1.2.5):** the receptor is read with `--read_pdb` (no ProDy needed) and Meeko adds the polar hydrogens; the crystal ligand needs explicit hydrogens before Meeko can prepare it; redocking passes when at least `MIN_PASSING_SEEDS` of the seeds reproduce the crystal pose within 2 Å.

In [12]:
#Installing the required packages

%pip install meeko==0.8.0
!apt-get update -qq
!apt-get install -y autodock-vina
%pip install rdkit
%pip -q install biopython
%pip install gemmi
%pip install prody
%pip install -q plip
from importlib.metadata import version
print("PLIP version:", version("plip"))
import Bio
print("Biopython version:", Bio.__version__)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
autodock-vina is already the newest version (1.2.5-2build1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
PLIP version: 3.0.1
Biopython version: 1.88


In [13]:
# Colab setup, when required: %pip install meeko rdkit biopython pandas
# Also install an AutoDock Vina executable and make `vina` available on PATH.
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
    if not (ROOT / 'ace_config.py').exists(): ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists(): raise FileNotFoundError('Open the Google Drive-synced ACE_Workflow folder in VS Code.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ace_config import *

import json, re, shutil, subprocess
import numpy as np, pandas as pd
from Bio.PDB import PDBIO, PDBParser, Select
from rdkit import Chem
from rdkit.Chem import AllChem

RAW_PDB = INPUTS['RECEPTOR_1O86']
assert RAW_PDB.exists(), f'Missing 1O86 receptor: {RAW_PDB}'
VINA = shutil.which('vina') or shutil.which('vina.exe')
MEEKO_RECEPTOR = shutil.which('mk_prepare_receptor.py')
MEEKO_LIGAND = shutil.which('mk_prepare_ligand.py')
if not all([VINA, MEEKO_RECEPTOR, MEEKO_LIGAND]):
    raise RuntimeError('Install AutoDock Vina and Meeko, then ensure vina/mk_prepare_* are on PATH.')
SEEDS, EXHAUSTIVENESS, NUM_MODES, BOX_PADDING_A = [7, 42, 2026], 16, 20, 8.0
MIN_PASSING_SEEDS = 2  # a single lucky seed is not evidence that the setup reproduces the crystal pose
_v = subprocess.run([VINA, '--version'], text=True, capture_output=True)
VINA_VERSION = ((_v.stdout or _v.stderr).strip().splitlines() or ['unknown'])[0]
print('Vina:', VINA_VERSION)
if re.search(r'\b1\.1\.', VINA_VERSION): print('WARNING: Vina 1.1.x detected; use Vina >= 1.2 (see requirements/README).')
DOCKING_CPU = CPU_WORKERS  # CPU-only Vina run; no GPU is requested

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Vina: AutoDock Vina v1.2.5


In [15]:
# Keep protein chain A and the catalytic Zn²⁺; remove LPR, chloride, glycerol, glycans and waters.
class ACEReceptorSelector(Select):
    def accept_chain(self, chain): return chain.id == 'A'
    def accept_residue(self, residue):
        # Bio.PDB residue.id is (hetflag, resseq, icode); the residue NAME is residue.get_resname().
        return residue.id[0] == ' ' or residue.get_resname().strip() == 'ZN'
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')  # one alternate location only

parser = PDBParser(QUIET=True)
structure = parser.get_structure('1O86', str(RAW_PDB))
receptor_pdb = PATHS['REDOCKING_RECEPTOR'] / '1o86_chainA_with_Zn.pdb'
io = PDBIO(); io.set_structure(structure); io.save(str(receptor_pdb), ACEReceptorSelector())
# Blank the alt-loc column so Meeko does not need --default_altloc.
receptor_lines = [(l[:16] + ' ' + l[17:]) if l.startswith(('ATOM', 'HETATM')) else l for l in receptor_pdb.read_text().splitlines()]
receptor_pdb.write_text('\n'.join(receptor_lines) + '\n')

# Limit unmatched-residue cleanup to residues well outside the crystallographic docking box.
lpr_residue = next((residue for residue in structure[0]['A'] if residue.get_resname().strip() == 'LPR'), None)
if lpr_residue is None: raise ValueError('Could not find crystallographic LPR in chain A to define the docking box.')
lpr_coords = np.asarray([atom.coord for atom in lpr_residue if atom.element.strip().upper() not in {'H', 'D'}])
if lpr_coords.size == 0: raise ValueError('Crystallographic LPR has no heavy-atom coordinates.')
meeko_box_center = lpr_coords.mean(axis=0)
meeko_box_size = np.maximum(18.0, np.ptp(lpr_coords, axis=0) + BOX_PADDING_A)

output_base = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor'
# --read_pdb avoids the ProDy dependency of -i; Meeko adds polar hydrogens and types the Zn.
# Unmatched residues within 4 Å of the box still fail preparation rather than being silently removed.
cmd = [MEEKO_RECEPTOR, '--read_pdb', str(receptor_pdb), '-o', str(output_base), '--write_pdbqt',
       '--box_center', *map(str, meeko_box_center), '--box_size', *map(str, meeko_box_size),
       '--delete_bad_res_from_box_radius', '4.0']
run = subprocess.run(cmd, text=True, capture_output=True)
if run.returncode: raise RuntimeError((run.stdout + run.stderr)[-3000:])
receptor_pdbqt = output_base.with_suffix('.pdbqt')
if not receptor_pdbqt.exists(): raise FileNotFoundError(f'Meeko did not create {receptor_pdbqt}\n{(run.stdout + run.stderr)[-1500:]}')

# Sanity checks: exactly one catalytic Zn and polar hydrogens present.
atom_types = [l[77:79].strip() for l in receptor_pdbqt.read_text().splitlines() if l.startswith(('ATOM', 'HETATM'))]
n_zn, n_hd = atom_types.count('Zn'), atom_types.count('HD')
if n_zn != 1: raise RuntimeError(f'Expected one Zn in the receptor PDBQT, found {n_zn}.')
if n_hd == 0: raise RuntimeError('Receptor PDBQT has no polar hydrogens (HD); check Meeko output.')
print(f'Receptor prepared: {receptor_pdbqt} ({len(atom_types)} atoms, {n_hd} polar H, {n_zn} Zn)')

Receptor prepared: /content/drive/MyDrive/ACE_Workflow/redocking_1o86/receptor/1o86_receptor.pdbqt (5610 atoms, 989 polar H, 1 Zn)


In [16]:
# Extract the crystallographic LPR pose, assign its known lisinopril bond orders, and preserve coordinates.
class LPRSelector(Select):
    def accept_residue(self, residue): return residue.resname.strip() == 'LPR' and residue.get_parent().id == 'A'
crystal_pdb = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.pdb'
io = PDBIO(); io.set_structure(structure); io.save(str(crystal_pdb), LPRSelector())
# Lisinopril (C21H31N3O5, 29 heavy atoms). The previous template was a different molecule (C22H34N4O4) and failed to match.
template = Chem.MolFromSmiles('NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O')
crystal_raw = Chem.MolFromPDBFile(str(crystal_pdb), removeHs=False, sanitize=False)
if crystal_raw is None or crystal_raw.GetNumAtoms() != template.GetNumHeavyAtoms():
    raise ValueError(f'LPR in chain A has {0 if crystal_raw is None else crystal_raw.GetNumAtoms()} atoms; lisinopril has {template.GetNumHeavyAtoms()}.')
crystal_mol = AllChem.AssignBondOrdersFromTemplate(template, crystal_raw)
Chem.SanitizeMol(crystal_mol)
crystal_sdf = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.sdf'
# Meeko needs explicit hydrogens; add them with coordinates (heavy-atom positions stay at the crystal pose).
crystal_h = Chem.AddHs(crystal_mol, addCoords=True)
writer = Chem.SDWriter(str(crystal_sdf)); writer.write(crystal_h); writer.close()
ligand_pdbqt = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.pdbqt'
run = subprocess.run([MEEKO_LIGAND, '-i', str(crystal_sdf), '-o', str(ligand_pdbqt)], text=True, capture_output=True)
if run.returncode: raise RuntimeError(run.stderr[-3000:])
coords = np.array([list(crystal_mol.GetConformer().GetAtomPosition(i)) for i in range(crystal_mol.GetNumAtoms()) if crystal_mol.GetAtomWithIdx(i).GetAtomicNum() > 1])
center, span = coords.mean(axis=0), np.ptp(coords, axis=0)  # ndarray.ptp was removed in NumPy 2
box = {'center_x': float(center[0]), 'center_y': float(center[1]), 'center_z': float(center[2]),
       'size_x': float(max(18.0, span[0] + BOX_PADDING_A)), 'size_y': float(max(18.0, span[1] + BOX_PADDING_A)), 'size_z': float(max(18.0, span[2] + BOX_PADDING_A)),
       'source': '1O86 chain A LPR (lisinopril) heavy-atom centroid'}
box_file = PATHS['REDOCKING_ANALYSIS'] / '1o86_docking_box.json'
box_file.write_text(json.dumps(box, indent=2))
box

[15:25:16] WARNING: More than one matching pattern found - picking one



{'center_x': 41.01362068965516,
 'center_y': 34.325206896551734,
 'center_z': 46.44120689655172,
 'size_x': 18.0,
 'size_y': 18.048000000000002,
 'size_z': 19.865000000000002,
 'source': '1O86 chain A LPR (lisinopril) heavy-atom centroid'}

In [18]:
runs = []
for seed in SEEDS:
    pose = PATHS['REDOCKING_DOCKING'] / f'LPR_redocked_seed_{seed}.pdbqt'
    log = PATHS['REDOCKING_DOCKING'] / f'LPR_redocked_seed_{seed}.log'
    cmd = [VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(ligand_pdbqt),
           '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
           '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
           '--cpu', str(DOCKING_CPU), '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    log.write_text(result.stdout + result.stderr)
    if result.returncode: raise RuntimeError(f'Seed {seed} failed: {(result.stdout + result.stderr)[-2000:]}')
    affinity = next((float(line.split()[3]) for line in pose.read_text(errors='replace').splitlines() if line.startswith('REMARK VINA RESULT')), np.nan)
    runs.append({'seed': seed, 'pose_pdbqt': str(pose), 'affinity_kcal_mol': affinity})
redock = pd.DataFrame(runs); display(redock)

,seed,pose_pdbqt,affinity_kcal_mol
0,7,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.960
1,42,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.905
2,2026,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.893


In [19]:
# Direct RMSD in the receptor frame; coordinates are not translated or superposed.
def pdbqt_heavy_coordinates(path):
    values, started = [], False
    for line in Path(path).read_text(errors='replace').splitlines():
        if line.startswith('MODEL'): started = True; continue
        if started and line.startswith('ENDMDL'): break
        if line.startswith(('ATOM', 'HETATM')) and line[77:79].strip().upper() not in {'H', 'HD', 'HS'}:
            values.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(values)
reference = pdbqt_heavy_coordinates(ligand_pdbqt)
def direct_rmsd(a, b): return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))
redock['heavy_atom_rmsd_A'] = redock.pose_pdbqt.map(lambda p: direct_rmsd(reference, pdbqt_heavy_coordinates(p)) if len(pdbqt_heavy_coordinates(p)) == len(reference) else np.nan)
redock['pass_2A'] = redock.heavy_atom_rmsd_A <= 2.0
n_pass = int(redock.pass_2A.sum()); redocking_passed = n_pass >= MIN_PASSING_SEEDS
best = redock.sort_values(['heavy_atom_rmsd_A', 'affinity_kcal_mol']).iloc[0]
best_pose = PATHS['REDOCKING_DOCKING'] / 'LPR_redocked_best.pdbqt'
shutil.copy2(best.pose_pdbqt, best_pose)
redock.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_summary.csv', index=False)
(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_metadata.json').write_text(json.dumps({'raw_pdb': str(RAW_PDB), 'raw_pdb_sha256': sha256_file(RAW_PDB), 'receptor_pdbqt': str(receptor_pdbqt), 'box': box, 'vina_version': VINA_VERSION, 'seeds': SEEDS, 'n_seeds_pass_2A': n_pass, 'min_passing_seeds': MIN_PASSING_SEEDS, 'redocking_passed': redocking_passed, 'median_rmsd_A': float(redock.heavy_atom_rmsd_A.median()), 'best_seed': int(best.seed), 'best_rmsd_A': float(best.heavy_atom_rmsd_A)}, indent=2))
display(redock.sort_values('heavy_atom_rmsd_A'))
print(f'{n_pass}/{len(redock)} seeds within 2 Å (median RMSD {redock.heavy_atom_rmsd_A.median():.2f} Å). ' + ('Redocking PASSED.' if redocking_passed else 'Redocking FAILED: resolve the setup before candidate docking.'))

,seed,pose_pdbqt,affinity_kcal_mol,heavy_atom_rmsd_A,pass_2A
0,7,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.960,9.442119,False
2,2026,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.893,9.458205,False
1,42,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.905,9.466482,False


0/3 seeds within 2 Å (median RMSD 9.46 Å). Redocking FAILED: resolve the setup before candidate docking.
